[Open in Colab](https://colab.research.google.com/github/Shineii86/MoeStickerBot/blob/main/notebooks/MoeStickerBotV0.ipynb)

# Moe Sticker Bot
Self‑host your Telegram sticker bot – import LINE / Kakao, create & manage stickers.


In [ ]:
#@title 📦 1. Setup Environment & Build Bot (Run Once)

import sys, time, subprocess, os, urllib.request, json, requests

print("✨ Ready!")

# ---------- System dependencies ----------
print("\nInstalling system packages...")
!apt-get update -qq 2>/dev/null
!apt-get install -y -qq imagemagick libarchive-tools ffmpeg curl gifsicle python3 exiv2 2>/dev/null
print("Core packages installed.")

# Go
url = "https://go.dev/dl/go1.21.5.linux-amd64.tar.gz"
print("Downloading Go...")
urllib.request.urlretrieve(url, "go.tar.gz")
!tar -C /usr/local -xzf go.tar.gz
os.environ['PATH'] += ":/usr/local/go/bin"
os.environ['GOPATH'] = "/root/go"
os.environ['GO111MODULE'] = "on"
!mkdir -p $GOPATH
print(f"Go {subprocess.getoutput('go version').split()[2]} installed.")

# Python helpers
print("\nInstalling Python helpers...")
!wget -q https://raw.githubusercontent.com/star-39/moe-sticker-bot/master/tools/msb_emoji.py -O /usr/local/bin/msb_emoji.py
!wget -q https://raw.githubusercontent.com/star-39/moe-sticker-bot/master/tools/msb_kakao_decrypt.py -O /usr/local/bin/msb_kakao_decrypt.py
!wget -q https://raw.githubusercontent.com/star-39/moe-sticker-bot/master/tools/msb_rlottie.py -O /usr/local/bin/msb_rlottie.py
!chmod +x /usr/local/bin/msb_emoji.py /usr/local/bin/msb_kakao_decrypt.py /usr/local/bin/msb_rlottie.py
print("Helpers installed.")

# Build the bot
print("\nBuilding moe-sticker-bot...")
!rm -rf moe-sticker-bot
!git clone --depth 1 https://github.com/star-39/moe-sticker-bot.git 2>&1 | grep -v "Cloning"
%cd moe-sticker-bot
!go mod download
!go build -o moe-sticker-bot cmd/moe-sticker-bot/main.go
if os.path.exists("moe-sticker-bot"):
    sz = os.path.getsize("moe-sticker-bot")/1024/1024
    print(f"Build complete — Binary: {sz:.1f} MB")
else:
    print("Build failed!")


In [ ]:
#@title ⚙️ 2. Configure & Launch Bot

# ---------- Configuration ----------
BOT_TOKEN = ""  #@param {type:"string"}
ENABLE_DB = False  #@param {type:"boolean"}
DB_ADDR = "localhost:3306"  #@param {type:"string"}
DB_USER = "moe_bot"  #@param {type:"string"}
DB_PASS = ""  #@param {type:"string"}
DB_NAME = "moe_sticker_bot"  #@param {type:"string"}
ENABLE_WEBAPP = False  #@param {type:"boolean"}
WEBAPP_PORT = 8080  #@param {type:"integer"}
NGROK_AUTHTOKEN = ""  #@param {type:"string"}
DATA_DIR = "moe_sticker_bot_data"  #@param {type:"string"}
LOG_LEVEL = "info"  #@param ["debug", "info", "warn", "error"]
HTTP_PROXY = ""  #@param {type:"string"}

if BOT_TOKEN:
    print(f"BOT_TOKEN = {BOT_TOKEN[:8]}...{BOT_TOKEN[-4:]}")
else:
    print("WARNING: BOT_TOKEN is missing!")

if ENABLE_WEBAPP and not NGROK_AUTHTOKEN:
    print("WebApp enabled but no ngrok token — disabled.")
    ENABLE_WEBAPP = False

# ---------- ngrok (if enabled) ----------
WEBAPP_URL = ""
if ENABLE_WEBAPP:
    print("\nSetting up ngrok tunnel...")
    if not os.path.exists("./ngrok"):
        !wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz && tar -xzf ngrok*.tgz && chmod +x ngrok
    !./ngrok config add-authtoken {NGROK_AUTHTOKEN}
    !pkill -f ngrok || true
    import subprocess, time, requests
    ngrok_proc = subprocess.Popen(["./ngrok", "http", str(WEBAPP_PORT), "--log", "stdout"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    for _ in range(10):
        try:
            r = requests.get("http://127.0.0.1:4040/api/tunnels")
            if r.status_code==200:
                tuns = r.json()['tunnels']
                if tuns:
                    WEBAPP_URL = tuns[0]['public_url']
                    print(f"ngrok URL: {WEBAPP_URL}")
                    break
        except:
            pass
        time.sleep(1)
    else:
        print("Could not retrieve ngrok URL.")
        ENABLE_WEBAPP = False

# ---------- Launch ----------
print("\nLaunching bot...")
if not BOT_TOKEN:
    print("ERROR: No BOT_TOKEN provided. Aborting.")
    sys.exit(1)

cmd_line = ["./moe-sticker-bot", f"--bot_token={BOT_TOKEN}", f"--log_level={LOG_LEVEL}", f"--data_dir={DATA_DIR}"]
if ENABLE_DB and DB_ADDR:
    cmd_line.extend([f"--db_addr={DB_ADDR}", f"--db_user={DB_USER}", f"--db_pass={DB_PASS}", f"--db_name={DB_NAME}"])
if ENABLE_WEBAPP and WEBAPP_URL:
    cmd_line.append(f"--webapp_url={WEBAPP_URL}")
    cmd_line.append(f"--webapp_listen_addr=0.0.0.0:{WEBAPP_PORT}")
if HTTP_PROXY:
    cmd_line.append(f"--http_proxy={HTTP_PROXY}")

log_out = open("bot_stdout.log", "w")
log_err = open("bot_stderr.log", "w")
process = subprocess.Popen(cmd_line, stdout=log_out, stderr=log_err)
time.sleep(3)
if process.poll() is None:
    print(f"Bot is RUNNING — PID {process.pid}")
    print("Send /start to your bot on Telegram!")
    if WEBAPP_URL:
        print(f"WebApp: {WEBAPP_URL}")
else:
    print("Bot exited immediately. Check bot_stderr.log.")
    !cat bot_stderr.log


In [ ]:
#@title 📜 3. Monitor & Control

ACTION = "View Logs"  #@param ["View Logs", "Stop Bot"]
LOG_TYPE = "stderr"  #@param ["stdout", "stderr"]
LINES = 30  #@param {type:"slider", min:10, max:100, step:10}

if ACTION == "View Logs":
    print(f"Last {LINES} lines of bot_{LOG_TYPE}.log:\n")
    !tail -n {LINES} bot_{LOG_TYPE}.log
else:
    print("Shutting down...")
    !pkill -f moe-sticker-bot && echo "Bot terminated" || echo "No bot running"
    !pkill -f ngrok && echo "ngrok terminated" || echo "No ngrok running"
    print("Cleanup complete.")


---
Made with ❤️ for the Sticker Community
